# EDA: S&P 500 Earnings Transcripts

Goal:
- Load `kurry/sp500_earnings_transcripts` with `datasets`.
- Print basic dataset information for quick inspection.


In [1]:
import os
from dotenv import load_dotenv

# Load .env and read HF_TOKEN from it
load_dotenv()

hf_token = os.getenv("HF_TOKEN")
if hf_token:
    print("HF_TOKEN loaded from .env (or environment).")
else:
    raise RuntimeError("HF_TOKEN not found. Please set HF_TOKEN in .env.")


HF_TOKEN loaded from .env (or environment).


In [2]:
from datasets import load_dataset
import pandas as pd

DATASET_ID = "kurry/sp500_earnings_transcripts"
DATASET_REVISION = os.getenv("HF_DATASET_REVISION")  # pin for reproducibility (commit hash / tag)
STRICT_REPRO = os.getenv("STRICT_REPRO", "0").strip() == "1"

hf_token = globals().get("hf_token", "")

ds = None
try:
    if STRICT_REPRO and not DATASET_REVISION:
        raise RuntimeError("STRICT_REPRO=1 but HF_DATASET_REVISION is not set. Set HF_DATASET_REVISION to pin the dataset version.")

    if DATASET_REVISION:
        ds = load_dataset(DATASET_ID, token=hf_token, revision=DATASET_REVISION)
    else:
        ds = load_dataset(DATASET_ID, token=hf_token)

    train_fp = getattr(ds.get("train", None), "_fingerprint", None)
    print(f"Loaded dataset: {DATASET_ID}")
    print(f"Dataset revision: {DATASET_REVISION}")
    print(f"Train fingerprint: {train_fp}")
    if not DATASET_REVISION:
        print("WARNING: HF_DATASET_REVISION not set; reruns may change if the dataset updates upstream.")
except Exception as e:
    print("Failed to load dataset. Check HF_TOKEN and your network.")
    print(type(e).__name__, str(e))


Loaded dataset: kurry/sp500_earnings_transcripts
Dataset revision: None
Train fingerprint: 7f18bf7871bd3ccd


In [3]:
if ds is None:
    print("Dataset is not available yet. Re-run the previous cell after network/login is fixed.")
else:
    split_names = list(ds.keys())
    print("Dataset object:", ds)
    print("Splits:", split_names)
    print()

    for split in split_names:
        subset = ds[split]
        print(f"[{split}] rows={subset.num_rows}, columns={len(subset.column_names)}")
        print("Columns:", subset.column_names)
        print("Features:", subset.features)
        print("-" * 80)


Dataset object: DatasetDict({
    train: Dataset({
        features: ['symbol', 'quarter', 'year', 'date', 'content', 'structured_content', 'company_name', 'company_id'],
        num_rows: 33362
    })
})
Splits: ['train']

[train] rows=33362, columns=8
Columns: ['symbol', 'quarter', 'year', 'date', 'content', 'structured_content', 'company_name', 'company_id']
Features: {'symbol': Value('string'), 'quarter': Value('int64'), 'year': Value('int64'), 'date': Value('string'), 'content': Value('string'), 'structured_content': List({'speaker': Value('string'), 'text': Value('string')}), 'company_name': Value('string'), 'company_id': Value('float64')}
--------------------------------------------------------------------------------


In [4]:
if ds is None:
    print("Dataset is not available yet.")
else:
    primary_split = list(ds.keys())[0]
    subset = ds[primary_split]
    sample_size = min(5, subset.num_rows)
    sample_df = subset.select(range(sample_size)).to_pandas()

    print(f"Preview from split: {primary_split} (first {sample_size} rows)")
    display(sample_df)


Preview from split: train (first 5 rows)


,symbol,quarter,year,date,content,structured_content,company_name,company_id
0,A,4,2020,2020-11-23 16:30:00,"Operator: Good afternoon, and welcome to the A...","[{'speaker': 'Operator', 'text': 'Good afterno...","Agilent Technologies, Inc.",154924.0
1,A,3,2020,2020-08-18 16:30:00,"Operator: Good afternoon, and welcome to the A...","[{'speaker': 'Operator', 'text': 'Good afterno...","Agilent Technologies, Inc.",154924.0
2,A,2,2020,2020-05-21 16:30:00,"Operator: Good afternoon, and welcome to the A...","[{'speaker': 'Operator', 'text': 'Good afterno...","Agilent Technologies, Inc.",154924.0
3,A,1,2020,2020-02-18 16:30:00,Operator: Good afternoon and welcome to the Ag...,"[{'speaker': 'Operator', 'text': 'Good afterno...","Agilent Technologies, Inc.",154924.0
4,A,4,2021,2021-11-22 16:30:00,Operator: Good afternoon and welcome to the Ag...,"[{'speaker': 'Operator', 'text': 'Good afterno...","Agilent Technologies, Inc.",154924.0


In [5]:
if ds is None:
    print("Dataset is not available yet.")
else:
    primary_split = list(ds.keys())[0]
    subset = ds[primary_split]
    inspect_rows = min(10000, subset.num_rows)
    inspect_df = subset.select(range(inspect_rows)).to_pandas()

    print(f"Missing value stats on first {inspect_rows} rows of split '{primary_split}':")
    missing = inspect_df.isna().sum().sort_values(ascending=False)
    missing_ratio = (missing / inspect_rows).round(4)
    missing_summary = pd.DataFrame({"missing_count": missing, "missing_ratio": missing_ratio})
    display(missing_summary)


Missing value stats on first 10000 rows of split 'train':


,missing_count,missing_ratio
symbol,0,0.0
quarter,0,0.0
year,0,0.0
date,0,0.0
content,0,0.0
structured_content,0,0.0
company_name,0,0.0
company_id,0,0.0


In [6]:
if ds is None:
    print("Dataset is not available yet.")
else:
    primary_split = list(ds.keys())[0]
    subset = ds[primary_split]
    inspect_rows = min(10000, subset.num_rows)
    inspect_df = subset.select(range(inspect_rows)).to_pandas()

    text_candidates = [
        c for c in inspect_df.columns
        if any(k in c.lower() for k in ["transcript", "text", "content"])
    ]

    if not text_candidates:
        text_candidates = [
            c for c in inspect_df.columns
            if inspect_df[c].dtype == "object"
        ]

    if not text_candidates:
        print("No text-like column found for length statistics.")
    else:
        text_col = text_candidates[0]
        lengths = inspect_df[text_col].fillna("").astype(str).str.len()
        print(f"Text length statistics on column '{text_col}' (first {inspect_rows} rows):")
        print(lengths.describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.99]))


Text length statistics on column 'content' (first 10000 rows):
count     10000.000000
mean      52352.695200
std       11776.456536
min           0.000000
25%       46472.250000
50%       52704.500000
75%       57875.250000
90%       64626.100000
99%       84516.960000
max      226464.000000
Name: content, dtype: float64


## Date-Key Split (Keep Structured Content + Company Name)

- Use `date` as the split key.
- Keep only `structured_content` and `company_name` in each split payload.


In [7]:
if ds is None:
    print("Dataset is not available yet.")
else:
    required_cols = ["date", "symbol", "year", "quarter", "company_name", "structured_content"]
    split_df = ds["train"].to_pandas()[required_cols].copy()
    split_df["date"] = pd.to_datetime(split_df["date"], errors="coerce")
    split_df["year"] = pd.to_numeric(split_df["year"], errors="coerce").astype("Int64")
    split_df["quarter"] = pd.to_numeric(split_df["quarter"], errors="coerce").astype("Int64")
    if split_df["year"].isna().any() or split_df["quarter"].isna().any():
        raise RuntimeError("Found missing year/quarter in transcript dataset. Refusing to proceed.")
    split_df["year"] = split_df["year"].astype(int)
    split_df["quarter"] = split_df["quarter"].astype(int)

    # Only drop invalid date rows; keep rows even if symbol/company_name is missing
    split_df = split_df.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)

    # Keep only data from 2020-05-01 to 2025-05-31 (exclusive upper bound: 2025-06-01)
    start_date = pd.Timestamp("2020-05-01")
    end_date_exclusive = pd.Timestamp("2025-06-01")
    split_df = split_df[(split_df["date"] >= start_date) & (split_df["date"] < end_date_exclusive)].copy()

    # Normalize ticker
    split_df["ticker"] = split_df["symbol"].astype("string").str.strip().str.upper()
    split_df = split_df.dropna(subset=["ticker"]).copy()

    rows_after_window = len(split_df)
    tickers_after_window = split_df["ticker"].nunique(dropna=True)

    # (1) Keep only tickers that appear in EVERY calendar year within the window
    required_years = list(range(2020, 2026))  # 2020..2025
    split_df["calendar_year"] = split_df["date"].dt.year
    years_by_ticker = split_df.groupby("ticker")["calendar_year"].agg(lambda s: set(int(x) for x in s.dropna().astype(int).tolist()))
    full_year_tickers = years_by_ticker.index[years_by_ticker.apply(lambda ys: set(required_years).issubset(ys))].tolist()
    split_df = split_df[split_df["ticker"].isin(full_year_tickers)].copy()

    rows_after_full_year = len(split_df)
    tickers_after_full_year = split_df["ticker"].nunique(dropna=True)

    # (2) Keep only tickers that also exist in WRDS metadata (Sp500_meta_data.csv)
    WRDS_META_PATH = "Sp500_meta_data.csv"
    wrds_tickers = (
        pd.read_csv(WRDS_META_PATH, usecols=["tic"], dtype={"tic": "string"})["tic"]
        .astype("string")
        .str.strip()
        .str.upper()
        .dropna()
        .unique()
        .tolist()
    )
    split_df = split_df[split_df["ticker"].isin(set(wrds_tickers))].copy()

    rows_after_wrds = len(split_df)
    tickers_after_wrds = split_df["ticker"].nunique(dropna=True)

    # Final schema (keep ticker/year/quarter for downstream merge)
    split_df = split_df[["date", "symbol", "ticker", "year", "quarter", "company_name", "structured_content"]].copy()

    date_key_series = split_df["date"].dt.strftime("%Y-%m-%d")
    date_keyed_data = (
        split_df.groupby(date_key_series, sort=True)[["symbol", "ticker", "company_name", "structured_content"]]
        .apply(lambda g: g.to_dict(orient="records"))
        .to_dict()
    )

    date_split_summary = (
        split_df.groupby(date_key_series, sort=True)
        .size()
        .rename_axis("date_key")
        .reset_index(name="row_count")
    )

    print(f"Date window: {start_date.date()} to {(end_date_exclusive - pd.Timedelta(days=1)).date()}")
    print(f"Required years (calendar): {required_years}")
    print(f"Rows after window filter: {rows_after_window}")
    print(f"Unique tickers after window: {tickers_after_window}")
    print(f"Rows after full-year filter: {rows_after_full_year}")
    print(f"Unique tickers after full-year: {tickers_after_full_year}")
    print(f"Rows after WRDS ticker filter: {rows_after_wrds}")
    print(f"Unique tickers after WRDS: {tickers_after_wrds}")
    print(f"Total unique date keys: {len(date_keyed_data)}")
    print("Columns kept in split_df:", split_df.columns.tolist())
    print("Missing company_name rows kept:", int(split_df["company_name"].isna().sum()))


Date window: 2020-05-01 to 2025-05-31
Required years (calendar): [2020, 2021, 2022, 2023, 2024, 2025]
Rows after window filter: 10759
Unique tickers after window: 578
Rows after full-year filter: 10011
Unique tickers after full-year: 493
Rows after WRDS ticker filter: 9432
Unique tickers after WRDS: 464
Total unique date keys: 954
Columns kept in split_df: ['date', 'symbol', 'ticker', 'year', 'quarter', 'company_name', 'structured_content']
Missing company_name rows kept: 0


In [8]:
if ds is None:
    print("Dataset is not available yet.")
else:
    print("Date split summary (head):")
    display(date_split_summary.head(10))
    print("Date split summary (tail):")
    display(date_split_summary.tail(10))

    print("Filtered dataset preview (4 columns):")
    display(split_df.head(5))

    sample_date_key = date_split_summary.iloc[-1]["date_key"]
    sample_records = date_keyed_data[sample_date_key][:2]
    print(f"Sample date key: {sample_date_key}")
    print(f"Records on that date: {len(date_keyed_data[sample_date_key])}")
    print("First 2 records (symbol/company_name/structured_content):")
    sample_records


Date split summary (head):


,date_key,row_count
0,2020-05-01,27
1,2020-05-02,3
2,2020-05-04,8
3,2020-05-05,31
4,2020-05-06,31
5,2020-05-07,33
6,2020-05-08,22
7,2020-05-09,9
8,2020-05-10,8
9,2020-05-11,8


Date split summary (tail):


,date_key,row_count
944,2025-05-01,64
945,2025-05-02,17
946,2025-05-05,11
947,2025-05-06,27
948,2025-05-07,24
949,2025-05-08,22
950,2025-05-09,1
951,2025-05-12,4
952,2025-05-14,1
953,2025-05-15,4


Filtered dataset preview (4 columns):


,date,symbol,ticker,year,quarter,company_name,structured_content
22604,2020-05-01 07:30:00,JCI,JCI,2020,2,Johnson Controls International plc,"[{'speaker': 'Operator', 'text': 'Welcome to J..."
22605,2020-05-01 08:00:00,MOH,MOH,2020,1,"Molina Healthcare, Inc.","[{'speaker': 'Operator', 'text': 'Good day, an..."
22608,2020-05-01 08:15:00,RJF,RJF,2020,2,"Raymond James Financial, Inc.","[{'speaker': 'Kristie Waugh', 'text': 'Good mo..."
22609,2020-05-01 08:30:00,HON,HON,2020,1,Honeywell International Inc.,"[{'speaker': 'Operator', 'text': 'Good day, la..."
22610,2020-05-01 08:30:00,CBOE,CBOE,2020,1,"Cboe Global Markets, Inc.","[{'speaker': 'Operator', 'text': 'Hello, and w..."


Sample date key: 2025-05-15
Records on that date: 4
First 2 records (symbol/company_name/structured_content):


In [9]:
split_df.head()

,date,symbol,ticker,year,quarter,company_name,structured_content
22604,2020-05-01 07:30:00,JCI,JCI,2020,2,Johnson Controls International plc,"[{'speaker': 'Operator', 'text': 'Welcome to J..."
22605,2020-05-01 08:00:00,MOH,MOH,2020,1,"Molina Healthcare, Inc.","[{'speaker': 'Operator', 'text': 'Good day, an..."
22608,2020-05-01 08:15:00,RJF,RJF,2020,2,"Raymond James Financial, Inc.","[{'speaker': 'Kristie Waugh', 'text': 'Good mo..."
22609,2020-05-01 08:30:00,HON,HON,2020,1,Honeywell International Inc.,"[{'speaker': 'Operator', 'text': 'Good day, la..."
22610,2020-05-01 08:30:00,CBOE,CBOE,2020,1,"Cboe Global Markets, Inc.","[{'speaker': 'Operator', 'text': 'Hello, and w..."


## Unique Ticker Metadata from WRDS (Sp500_meta_data.csv)

Load S&P 500 company metadata from local WRDS file `Sp500_meta_data.csv`.
This replaces the previous approach using external HuggingFace datasets.

GICS codes are mapped to human-readable names:
- `gsector` (2-digit) → Sector name (e.g. 35 → 'Health Care')
- `gsubind` (8-digit) → Industry name (e.g. 35101010 → 'Health Care Equipment')


In [10]:
# Load WRDS Sp500_meta_data.csv (local file)
WRDS_META_PATH = "Sp500_meta_data.csv"

# GICS Sector code to name mapping
GICS_SECTOR_MAP = {
    10: "Energy",
    15: "Materials",
    20: "Industrials",
    25: "Consumer Discretionary",
    30: "Consumer Staples",
    35: "Health Care",
    40: "Financials",
    45: "Information Technology",
    50: "Communication Services",
    55: "Utilities",
    60: "Real Estate"
}

# GICS Sub-Industry code to name mapping (comprehensive)
GICS_SUBIND_MAP = {
    10101010: "Oil & Gas Drilling", 10101020: "Oil & Gas Equipment & Services",
    10102010: "Integrated Oil & Gas", 10102020: "Oil & Gas Exploration & Production",
    10102030: "Oil & Gas Refining & Marketing", 10102040: "Oil & Gas Storage & Transportation",
    15101010: "Commodity Chemicals", 15101020: "Diversified Chemicals",
    15101030: "Fertilizers & Agricultural Chemicals", 15101040: "Industrial Gases",
    15101050: "Specialty Chemicals", 15102010: "Construction Materials",
    15103010: "Metal, Glass & Plastic Containers", 15103020: "Paper & Plastic Packaging Products & Materials",
    15104020: "Diversified Metals & Mining", 15104025: "Copper", 15104030: "Gold",
    15104050: "Steel", 15105010: "Forest Products", 15105020: "Paper Products",
    20101010: "Aerospace & Defense", 20102010: "Building Products",
    20103010: "Construction & Engineering", 20104010: "Electrical Components & Equipment",
    20104020: "Heavy Electrical Equipment", 20105010: "Industrial Conglomerates",
    20106010: "Construction Machinery & Heavy Transportation Equipment",
    20106015: "Agricultural & Farm Machinery", 20106020: "Industrial Machinery & Supplies & Components",
    20107010: "Trading Companies & Distributors", 20201050: "Environmental & Facilities Services",
    20201060: "Office Services & Supplies", 20201070: "Diversified Support Services",
    20202010: "Human Resource & Employment Services", 20202020: "Research & Consulting Services",
    20202030: "Data Processing & Outsourced Services", 20301010: "Air Freight & Logistics",
    20302010: "Passenger Airlines", 20303010: "Marine Transportation",
    20304010: "Rail Transportation", 20304020: "Cargo Ground Transportation",
    25101010: "Auto Parts & Equipment", 25101020: "Tires & Rubber",
    25102010: "Automobile Manufacturers", 25201010: "Consumer Electronics",
    25201020: "Home Furnishings", 25201030: "Homebuilding", 25201040: "Household Appliances",
    25201050: "Housewares & Specialties", 25202010: "Leisure Products",
    25203010: "Apparel, Accessories & Luxury Goods", 25203020: "Footwear",
    25301010: "Casinos & Gaming", 25301020: "Hotels, Resorts & Cruise Lines",
    25301030: "Leisure Facilities", 25301040: "Restaurants",
    25302010: "Education Services", 25302020: "Specialized Consumer Services",
    25401025: "Cable & Satellite", 25401030: "Movies & Entertainment",
    25502020: "Internet & Direct Marketing Retail", 25503010: "Department Stores",
    25503020: "General Merchandise Stores", 25503030: "Broadline Retail",
    25504010: "Apparel Retail", 25504020: "Computer & Electronics Retail",
    25504030: "Home Improvement Retail", 25504040: "Other Specialty Retail",
    25504050: "Automotive Retail", 25504060: "Homefurnishing Retail",
    30101010: "Drug Retail", 30101020: "Food Distributors", 30101030: "Food Retail",
    30101040: "Consumer Staples Merchandise Retail", 30201010: "Brewers",
    30201020: "Distillers & Vintners", 30201030: "Soft Drinks & Non-alcoholic Beverages",
    30202010: "Agricultural Products & Services", 30202030: "Packaged Foods & Meats",
    30203010: "Tobacco", 30301010: "Household Products", 30302010: "Personal Care Products",
    35101010: "Health Care Equipment", 35101020: "Health Care Supplies",
    35102010: "Health Care Distributors", 35102015: "Health Care Services",
    35102020: "Health Care Facilities", 35102030: "Managed Health Care",
    35103010: "Health Care Technology", 35201010: "Biotechnology",
    35202010: "Pharmaceuticals", 35203010: "Life Sciences Tools & Services",
    40101010: "Diversified Banks", 40101015: "Regional Banks",
    40102010: "Diversified Financial Services", 40201020: "Multi-Sector Holdings",
    40201030: "Specialized Finance", 40201040: "Commercial & Residential Mortgage Finance",
    40202010: "Consumer Finance", 40203010: "Asset Management & Custody Banks",
    40203020: "Investment Banking & Brokerage", 40203030: "Diversified Capital Markets",
    40203040: "Financial Exchanges & Data", 40204010: "Mortgage REITs",
    40301010: "Insurance Brokers", 40301020: "Life & Health Insurance",
    40301030: "Multi-line Insurance", 40301040: "Property & Casualty Insurance",
    40301050: "Reinsurance", 40402010: "Transaction & Payment Processing Services",
    45101010: "Internet Services & Infrastructure", 45102010: "IT Consulting & Other Services",
    45102030: "Data Processing & Outsourced Services", 45103010: "Application Software",
    45103020: "Systems Software", 45201010: "Networking Equipment",
    45201020: "Communications Equipment", 45202010: "Technology Hardware, Storage & Peripherals",
    45202030: "Electronic Equipment & Instruments", 45203010: "Electronic Components",
    45203015: "Electronic Manufacturing Services", 45203020: "Technology Distributors",
    45301010: "Semiconductor Materials & Equipment", 45301020: "Semiconductors",
    50101010: "Alternative Carriers", 50101020: "Integrated Telecommunication Services",
    50102010: "Wireless Telecommunication Services", 50201030: "Cable & Satellite",
    50202010: "Movies & Entertainment", 50202020: "Interactive Home Entertainment",
    50203010: "Interactive Media & Services",
    55101010: "Electric Utilities", 55102010: "Gas Utilities", 55103010: "Multi-Utilities",
    55104010: "Water Utilities", 55105010: "Independent Power Producers & Energy Traders",
    55105020: "Renewable Electricity",
    60101010: "Diversified REITs", 60102010: "Industrial REITs", 60102020: "Hotel & Resort REITs",
    60102030: "Office REITs", 60102040: "Health Care REITs",
    60102050: "Multi-Family Residential REITs", 60102060: "Residential REITs",
    60102070: "Retail REITs", 60102080: "Other Specialized REITs",
    60103010: "Telecom Tower REITs", 60103020: "Data Center REITs",
    60103030: "Self-Storage REITs", 60201020: "Real Estate Operating Companies",
    60201030: "Real Estate Development", 60201040: "Real Estate Services",
}

# Add missing sub-industry codes observed in Sp500_meta_data.csv (to keep industry names non-null)
GICS_SUBIND_MAP.update(
    {
        20304030: "Cargo Ground Transportation",
        20304040: "Passenger Ground Transportation",
        25501010: "Distributors",
        40201060: "Transaction & Payment Processing Services",
        45203030: "Technology Distributors",
        50201010: "Advertising",
        50201020: "Broadcasting",
        50201040: "Publishing",
        60102510: "Industrial REITs",
        60104010: "Office REITs",
        60105010: "Health Care REITs",
        60106010: "Multi-Family Residential REITs",
        60106020: "Single-Family Residential REITs",
        60107010: "Retail REITs",
        60108010: "Other Specialized REITs",
        60108020: "Self-Storage REITs",
        60108030: "Telecom Tower REITs",
        60108040: "Timber REITs",
        60108050: "Data Center REITs",
    }
)

# Load WRDS metadata CSV with dtype enforcement
wrds_df = pd.read_csv(
    WRDS_META_PATH,
    dtype={
        "tic": "string",
        "conm": "string",
        "gsector": "Int64",
        "gsubind": "Int64",
    },
    parse_dates=["datadate"],
)

# Normalize ticker and GICS codes
wrds_df["tic"] = wrds_df["tic"].astype("string").str.strip().str.upper()
wrds_df["gsector"] = pd.to_numeric(wrds_df["gsector"], errors="coerce").astype("Int64")
wrds_df["gsubind"] = pd.to_numeric(wrds_df["gsubind"], errors="coerce").astype("Int64")

# Map GICS codes to names on full history (for time-aligned merge)
# Sp500_meta_data.csv is treated as the authoritative source for gsector/gsubind codes.
wrds_df["sector"] = wrds_df["gsector"].map(GICS_SECTOR_MAP)
if wrds_df["sector"].isna().any():
    raise RuntimeError("Unexpected missing sector mapping from gsector. Please verify GICS_SECTOR_MAP or source data.")

# Industry: keep authoritative WRDS gsubind code as the primary label (GICS_########).
# Optionally attach a best-effort human-readable name for convenience.
wrds_df["industry_code"] = wrds_df["gsubind"].astype("Int64").astype("string").str.zfill(8)
wrds_df["industry"] = "GICS_" + wrds_df["industry_code"]
wrds_df["industry_name"] = wrds_df["gsubind"].map(GICS_SUBIND_MAP)

# Build time-aligned metadata table (keep all historical rows)
wrds_time_df = (
    wrds_df[["tic", "conm", "datadate", "gsector", "gsubind", "industry_code", "sector", "industry", "industry_name"]]
    .rename(columns={"tic": "ticker", "conm": "company"})
    .dropna(subset=["ticker", "datadate"])
    .sort_values(["datadate", "ticker"])
    .reset_index(drop=True)
)

# Create ticker -> metadata lookup (latest record per ticker) for preview/summary
wrds_latest = (
    wrds_time_df.sort_values(["datadate", "ticker"])
    .drop_duplicates(subset=["ticker"], keep="last")
)

ticker_metadata_df = (
    wrds_latest[["ticker", "company", "sector", "industry"]]
    .dropna(subset=["ticker"])
    .drop_duplicates(subset=["ticker"], keep="last")
    .sort_values("ticker")
    .reset_index(drop=True)
)

print(f"Loaded WRDS metadata from: {WRDS_META_PATH}")
print(f"Unique tickers: {ticker_metadata_df['ticker'].nunique()}")
print(f"Sectors covered: {ticker_metadata_df['sector'].nunique()}")
print(f"Industries covered: {ticker_metadata_df['industry'].nunique()}")
print(f"Tickers with industry_name mapped: {int(ticker_metadata_df['ticker'].isin(wrds_df.loc[wrds_df['industry_name'].notna(), 'tic'].unique()).sum())}")
print(f"Missing sector values: {ticker_metadata_df['sector'].isna().sum()}")
print(f"Missing industry values: {ticker_metadata_df['industry'].isna().sum()}")


Loaded WRDS metadata from: Sp500_meta_data.csv
Unique tickers: 499
Sectors covered: 11
Industries covered: 125
Tickers with industry_name mapped: 499
Missing sector values: 0
Missing industry values: 0


In [11]:
print("Unique ticker metadata preview:")
display(ticker_metadata_df.head(20))

# Check for missing sector/industry mappings
missing_sector = ticker_metadata_df["sector"].isna().sum()
missing_industry = ticker_metadata_df["industry"].isna().sum()
if missing_sector > 0 or missing_industry > 0:
    print(f"Tickers with missing sector: {missing_sector}")
    print(f"Tickers with missing industry: {missing_industry}")
    display(ticker_metadata_df[ticker_metadata_df["sector"].isna() | ticker_metadata_df["industry"].isna()].head(10))
else:
    print("All tickers have sector and industry mappings.")


Unique ticker metadata preview:


,ticker,company,sector,industry
0,A,AGILENT TECHNOLOGIES INC,Health Care,GICS_35203010
1,AAPL,APPLE INC,Information Technology,GICS_45202030
2,ABBV,ABBVIE INC,Health Care,GICS_35201010
3,ABNB,AIRBNB INC,Consumer Discretionary,GICS_25301020
4,ABT,ABBOTT LABORATORIES,Health Care,GICS_35101010
5,ACGL,ARCH CAPITAL GROUP LTD,Financials,GICS_40301040
6,ACN,ACCENTURE PLC,Information Technology,GICS_45102010
7,ADBE,ADOBE INC,Information Technology,GICS_45103010
8,ADI,ANALOG DEVICES INC,Information Technology,GICS_45301020
9,ADM,ARCHER-DANIELS-MIDLAND CO,Consumer Staples,GICS_30202010


All tickers have sector and industry mappings.


## Merge `split_df` with `ticker_metadata_df`

- Merge on `split_df.symbol == ticker_metadata_df.ticker`
- Fill missing `company_name` from metadata `company`
- Build final ticker-keyed table with `Company_name, sector, industry, date, structured_content`


In [12]:
required_vars = ["split_df", "wrds_time_df"]
missing_vars = [v for v in required_vars if v not in globals()]

if missing_vars:
    print(f"Missing variables: {missing_vars}. Please run previous cells first.")
else:
    merge_base_df = split_df.copy()
    if "ticker" not in merge_base_df.columns:
        merge_base_df["ticker"] = merge_base_df["symbol"].astype("string").str.strip().str.upper()
    else:
        merge_base_df["ticker"] = merge_base_df["ticker"].astype("string").str.strip().str.upper()
    merge_base_df["date"] = pd.to_datetime(merge_base_df["date"], errors="coerce")

    # Split into valid/invalid rows for merge_asof requirements
    valid_mask = merge_base_df["ticker"].notna() & merge_base_df["date"].notna()
    valid_df = merge_base_df.loc[valid_mask].copy()
    invalid_df = merge_base_df.loc[~valid_mask].copy()

    # Sort for merge_asof
    valid_df = valid_df.sort_values(["date", "ticker"], kind="mergesort").reset_index(drop=True)

    wrds_time_df = wrds_time_df.copy()
    wrds_time_df["ticker"] = wrds_time_df["ticker"].astype("string").str.strip().str.upper()
    wrds_time_df["datadate"] = pd.to_datetime(wrds_time_df["datadate"], errors="coerce")
    wrds_time_df = (
        wrds_time_df.dropna(subset=["ticker", "datadate"])
        .sort_values(["datadate", "ticker"], kind="mergesort")
        .reset_index(drop=True)
    )

    # Time-aligned merge: use the most recent WRDS record at or before each transcript date
    merged_valid = pd.merge_asof(
        valid_df,
        wrds_time_df,
        left_on="date",
        right_on="datadate",
        by="ticker",
        direction="backward",
        allow_exact_matches=True,
    )

    # High-stakes safety: do NOT forward-fill from future metadata.
    # If backward asof cannot find a record (datadate is NA), fail fast.
    no_asof_match = merged_valid["datadate"].isna()
    if bool(no_asof_match.any()):
        bad = merged_valid.loc[no_asof_match, ["ticker", "date"]].head(20)
        raise RuntimeError(
            "merge_asof produced rows with no <=datadate match. "
            "This indicates missing historical metadata for some (ticker, date) pairs. "
            "Refusing to fill from future/latest records. Sample:\n" + bad.to_string(index=False)
        )

    if len(invalid_df) > 0:
        raise RuntimeError(f"Found {len(invalid_df)} rows with invalid ticker/date after filtering. Refusing to proceed.")

    merged_df = merged_valid

    # Validate sector/industry are present and non-empty
    sector_bad = merged_df["sector"].isna() | merged_df["sector"].astype(str).str.strip().eq("")
    industry_bad = merged_df["industry"].isna() | merged_df["industry"].astype(str).str.strip().eq("")
    if bool((sector_bad | industry_bad).any()):
        raise RuntimeError("Sector/industry contains missing/empty values after WRDS merge. Refusing to proceed.")

    # Restore sort order
    merged_df = merged_df.sort_values(["ticker", "date"], kind="mergesort").reset_index(drop=True)

    # Fill Company_name with priority: original company_name > WRDS company
    company_name_clean = merged_df["company_name"].replace(r"^\s*$", pd.NA, regex=True)
    merged_df["Company_name"] = company_name_clean.combine_first(merged_df["company"])

    # Fallback: if still missing, use ticker as Company_name (last resort)
    still_missing_mask = merged_df["Company_name"].isna()
    merged_df.loc[still_missing_mask, "Company_name"] = merged_df.loc[still_missing_mask, "ticker"]

    final_ticker_df = merged_df[[
        "ticker",
        "Company_name",
        "sector",
        "industry",
        "industry_code",
        "industry_name",
        "gsector",
        "gsubind",
        "datadate",
        "year",
        "quarter",
        "date",
        "structured_content",
    ]].copy()
    final_ticker_df = final_ticker_df.rename(columns={"datadate": "wrds_datadate"})
    final_ticker_df = final_ticker_df.sort_values(["ticker", "date"], kind="mergesort").reset_index(drop=True)

    final_ticker_table = final_ticker_df.set_index("ticker")[["Company_name", "sector", "industry", "date", "structured_content"]]

    missing_company_before = int(company_name_clean.isna().sum())
    missing_company_after = int(final_ticker_df["Company_name"].isna().sum())
    missing_sector = int(final_ticker_df["sector"].isna().sum())
    missing_industry = int(final_ticker_df["industry"].isna().sum())
    industry_code_fallback = int(final_ticker_df["industry"].astype("string").str.startswith("GICS_").sum())

    print(f"Rows in final table: {len(final_ticker_table)}")
    print(f"Unique ticker keys: {final_ticker_df['ticker'].nunique(dropna=True)}")
    print(f"Missing Company_name before fill: {missing_company_before}")
    print(f"Missing Company_name after fill: {missing_company_after}")
    print(f"Missing sector after merge: {missing_sector}")
    print(f"Missing industry after merge: {missing_industry}")
    print(f"Industry filled via code fallback (GICS_*): {industry_code_fallback}")


Rows in final table: 9432
Unique ticker keys: 464
Missing Company_name before fill: 0
Missing Company_name after fill: 0
Missing sector after merge: 0
Missing industry after merge: 0
Industry filled via code fallback (GICS_*): 9432


In [13]:
if "final_ticker_table" not in globals():
    print("`final_ticker_table` not found. Run previous merge cell first.")
else:
    print("Final ticker-keyed table preview:")
    display(final_ticker_table.head(20))

    unresolved_company = final_ticker_df[final_ticker_df["Company_name"].isna()]
    print(f"Rows still missing Company_name: {len(unresolved_company)}")
    if len(unresolved_company) > 0:
        display(unresolved_company[["ticker", "date"]].head(20))


Final ticker-keyed table preview:


,Company_name,sector,industry,date,structured_content
ticker,,,,,
A,"Agilent Technologies, Inc.",Health Care,GICS_35203010,2020-05-21 16:30:00,"[{'speaker': 'Operator', 'text': 'Good afterno..."
A,"Agilent Technologies, Inc.",Health Care,GICS_35203010,2020-08-18 16:30:00,"[{'speaker': 'Operator', 'text': 'Good afterno..."
A,"Agilent Technologies, Inc.",Health Care,GICS_35203010,2020-11-23 16:30:00,"[{'speaker': 'Operator', 'text': 'Good afterno..."
A,"Agilent Technologies, Inc.",Health Care,GICS_35203010,2021-02-16 16:30:00,"[{'speaker': 'Operator', 'text': 'Good afterno..."
A,"Agilent Technologies, Inc.",Health Care,GICS_35203010,2021-05-25 16:30:00,"[{'speaker': 'Operator', 'text': 'Good afterno..."
A,"Agilent Technologies, Inc.",Health Care,GICS_35203010,2021-08-17 16:30:00,"[{'speaker': 'Operator', 'text': 'Good afterno..."
A,"Agilent Technologies, Inc.",Health Care,GICS_35203010,2021-11-22 16:30:00,"[{'speaker': 'Operator', 'text': 'Good afterno..."
A,"Agilent Technologies, Inc.",Health Care,GICS_35203010,2022-02-22 16:30:00,"[{'speaker': 'Operator', 'text': 'Hello. And w..."
A,"Agilent Technologies, Inc.",Health Care,GICS_35203010,2022-05-24 16:30:00,"[{'speaker': 'Operator', 'text': 'Good afterno..."


Rows still missing Company_name: 0


## Final Sector/Industry Standardization (WRDS-based)

Standardize sector and industry labels using WRDS data as the primary source.
For tickers not in WRDS, manual overrides are applied.


In [14]:
if "final_ticker_df" not in globals():
    print("`final_ticker_df` not found. Run previous cells first.")
else:
    std_df = final_ticker_df.copy()
    std_df["ticker"] = std_df["ticker"].astype("string").str.strip().str.upper()

    # Standardize labels (already WRDS/GICS-based from prior merge step)
    std_df["sector_std"] = std_df["sector"].astype("string").str.strip().replace("", pd.NA)
    std_df["industry_std"] = std_df["industry"].astype("string").str.strip().replace("", pd.NA)

    if std_df["sector_std"].isna().any() or std_df["industry_std"].isna().any():
        raise RuntimeError("Found missing sector/industry after WRDS merge. Refusing to proceed.")

    std_df["classification_source"] = "wrds_official_codes"
    std_df.loc[std_df["sector_std"].astype("string").str.startswith("GICS_SECTOR_"), "classification_source"] = "wrds_sector_code_fallback"

    std_df["classification_confidence"] = "high"
    std_df.loc[std_df["classification_source"].isin(["wrds_sector_code_fallback"]), "classification_confidence"] = "medium"

    final_ticker_std_df = std_df.copy()
    final_ticker_std_table = final_ticker_std_df.set_index("ticker")[["Company_name", "sector_std", "industry_std", "date", "structured_content", "classification_source", "classification_confidence"]]

    print("Standardization complete.")
    print(f"Rows: {len(final_ticker_std_df)}")
    print(f"Unique tickers: {final_ticker_std_df['ticker'].nunique()}")
    print(f"Rows with industry_name available: {int(final_ticker_std_df['industry_name'].notna().sum())}")



Standardization complete.
Rows: 9432
Unique tickers: 464
Rows with industry_name available: 9432


In [15]:
if "final_ticker_std_df" not in globals():
    print("`final_ticker_std_df` not found.")
else:
    sec_m = final_ticker_std_df["sector_std"].isna() | final_ticker_std_df["sector_std"].astype(str).str.strip().eq("")
    ind_m = final_ticker_std_df["industry_std"].isna() | final_ticker_std_df["industry_std"].astype(str).str.strip().eq("")

    print(f"Missing sector_std rows: {int(sec_m.sum())}")
    print(f"Missing industry_std rows: {int(ind_m.sum())}")
    print(f"Missing std unique tickers: {final_ticker_std_df.loc[sec_m | ind_m, 'ticker'].nunique()}")

    print("classification_source distribution:")
    display(final_ticker_std_df["classification_source"].value_counts())



Missing sector_std rows: 0
Missing industry_std rows: 0
Missing std unique tickers: 0
classification_source distribution:


classification_source
wrds_official_codes    9432
Name: count, dtype: int64

## Final Complete Dataset

Build a single final table and validate missing values for key columns.


In [16]:
if "final_ticker_std_df" not in globals():
    print("`final_ticker_std_df` not found. Run previous cells first.")
else:
    final_dataset_df = final_ticker_std_df.copy()
    final_dataset_df["ticker"] = final_dataset_df["ticker"].astype("string").str.strip().str.upper()
    final_dataset_df["Company_name"] = final_dataset_df["Company_name"].replace(r"^\s*$", pd.NA, regex=True)
    final_dataset_df["date"] = pd.to_datetime(final_dataset_df["date"], errors="coerce")

    final_dataset_df = final_dataset_df.assign(
        sector=final_dataset_df["sector_std"],
        industry=final_dataset_df["industry_std"],
    )[[
        "ticker",
        "Company_name",
        "sector",
        "industry",
        "industry_name",
        "gsector",
        "gsubind",
        "industry_code",
        "wrds_datadate",
        "year",
        "quarter",
        "date",
        "structured_content",
        "classification_source",
        "classification_confidence",
    ]]
    final_dataset_df = final_dataset_df.sort_values(["date", "ticker"]).reset_index(drop=True)

    # Validations (fail fast)
    leakage_mask = final_dataset_df["wrds_datadate"].notna() & (final_dataset_df["wrds_datadate"] > final_dataset_df["date"])
    if bool(leakage_mask.any()):
        bad = final_dataset_df.loc[leakage_mask, ["ticker", "date", "wrds_datadate"]].head(20)
        raise RuntimeError("Found wrds_datadate > transcript date (time leakage). Sample:\n" + bad.to_string(index=False))

    required_years = list(range(2020, 2026))
    years_by_ticker = final_dataset_df.groupby("ticker")["date"].agg(lambda s: set(int(x) for x in pd.to_datetime(s, errors='coerce').dt.year.dropna().astype(int).tolist()))
    missing_year_tickers = years_by_ticker.index[years_by_ticker.apply(lambda ys: not set(required_years).issubset(ys))].tolist()
    if missing_year_tickers:
        raise RuntimeError(f"Found tickers missing required years after filtering: {missing_year_tickers[:20]} (showing up to 20)")

    key_missing = final_dataset_df[["ticker", "sector", "industry", "year", "quarter", "date", "structured_content", "gsector", "gsubind", "wrds_datadate"]].isna().sum()
    print("Missing values in key columns:")
    display(key_missing.to_frame("missing_count"))
    if int(key_missing.sum()) != 0:
        raise RuntimeError("Final dataset contains missing values in required columns. Refusing to proceed.")

    print(f"Total rows: {len(final_dataset_df)}")
    print(f"Unique tickers: {final_dataset_df['ticker'].nunique(dropna=True)}")
    print(f"Date range: {final_dataset_df['date'].min()} -> {final_dataset_df['date'].max()}")
    display(final_dataset_df.head(20))


Missing values in key columns:


,missing_count
ticker,0
sector,0
industry,0
year,0
quarter,0
date,0
structured_content,0
gsector,0
gsubind,0
wrds_datadate,0


Total rows: 9432
Unique tickers: 464
Date range: 2020-05-01 07:30:00 -> 2025-05-15 16:30:00


,ticker,Company_name,sector,industry,industry_name,gsector,gsubind,industry_code,wrds_datadate,year,quarter,date,structured_content,classification_source,classification_confidence
0,JCI,Johnson Controls International plc,Industrials,GICS_20102010,Building Products,20,20102010,20102010,2020-03-31,2020,2,2020-05-01 07:30:00,"[{'speaker': 'Operator', 'text': 'Welcome to J...",wrds_official_codes,high
1,MOH,"Molina Healthcare, Inc.",Health Care,GICS_35102030,Managed Health Care,35,35102030,35102030,2020-03-31,2020,1,2020-05-01 08:00:00,"[{'speaker': 'Operator', 'text': 'Good day, an...",wrds_official_codes,high
2,RJF,"Raymond James Financial, Inc.",Financials,GICS_40203020,Investment Banking & Brokerage,40,40203020,40203020,2020-03-31,2020,2,2020-05-01 08:15:00,"[{'speaker': 'Kristie Waugh', 'text': 'Good mo...",wrds_official_codes,high
3,AON,Aon plc,Financials,GICS_40301010,Insurance Brokers,40,40301010,40301010,2020-03-31,2020,1,2020-05-01 08:30:00,"[{'speaker': 'Operator', 'text': 'Good morning...",wrds_official_codes,high
4,CBOE,"Cboe Global Markets, Inc.",Financials,GICS_40203040,Financial Exchanges & Data,40,40203040,40203040,2020-03-31,2020,1,2020-05-01 08:30:00,"[{'speaker': 'Operator', 'text': 'Hello, and w...",wrds_official_codes,high
5,CHTR,"Charter Communications, Inc.",Communication Services,GICS_50201030,Cable & Satellite,50,50201030,50201030,2020-03-31,2020,1,2020-05-01 08:30:00,"[{'speaker': 'Operator', 'text': 'Ladies and g...",wrds_official_codes,high
6,CL,Colgate-Palmolive Company,Consumer Staples,GICS_30301010,Household Products,30,30301010,30301010,2020-03-31,2020,1,2020-05-01 08:30:00,"[{'speaker': 'Operator', 'text': 'Good morning...",wrds_official_codes,high
7,HON,Honeywell International Inc.,Industrials,GICS_20105010,Industrial Conglomerates,20,20105010,20105010,2020-03-31,2020,1,2020-05-01 08:30:00,"[{'speaker': 'Operator', 'text': 'Good day, la...",wrds_official_codes,high
8,ABBV,AbbVie Inc.,Health Care,GICS_35201010,Biotechnology,35,35201010,35201010,2020-03-31,2020,1,2020-05-01 09:00:00,"[{'speaker': 'Operator', 'text': 'Good morning...",wrds_official_codes,high
9,COR,"Cencora, Inc.",Health Care,GICS_35102010,Health Care Distributors,35,35102010,35102010,2020-03-31,2020,1,2020-05-01 09:00:00,"[{'speaker': 'Operator', 'text': 'Greetings, a...",wrds_official_codes,high


In [17]:
# Save final_dataset_df to CSV and Parquet
if "final_dataset_df" not in globals():
    print("`final_dataset_df` not found. Run previous cells first.")
else:
    output_csv = "final_dataset.csv"
    final_dataset_df.to_csv(output_csv, index=False, encoding="utf-8-sig")
    print(f"Saved final_dataset_df to: {output_csv}")
    print(f"Total rows exported: {len(final_dataset_df)}")

    output_parquet = "final_dataset.parquet"
    try:
        final_dataset_df.to_parquet(output_parquet, index=False)
        print(f"Saved final_dataset_df to: {output_parquet}")
    except Exception as e:
        print("Failed to save parquet. Install pyarrow or fastparquet if needed.")
        print(type(e).__name__, str(e))

    # Write an auditable run manifest for reproducibility and governance
    import datetime
    import hashlib
    import json
    import os
    import platform
    import sys
    import subprocess
    import datasets as hf_datasets

    def sha256_file(path: str) -> str:
        h = hashlib.sha256()
        with open(path, "rb") as f:
            for chunk in iter(lambda: f.read(1024 * 1024), b""):
                h.update(chunk)
        return h.hexdigest()

    git_head = None
    try:
        git_head = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
    except Exception:
        git_head = None

    wrds_meta_path = "Sp500_meta_data.csv"
    wrds_meta_sha256 = sha256_file(wrds_meta_path)

    csv_sha256 = sha256_file(output_csv)
    parquet_sha256 = sha256_file(output_parquet) if os.path.exists(output_parquet) else None

    ds_id = globals().get("DATASET_ID")
    ds_rev = globals().get("DATASET_REVISION")
    ds_obj = globals().get("ds")
    train_fp = None
    try:
        train_fp = getattr(ds_obj.get("train", None), "_fingerprint", None) if ds_obj is not None else None
    except Exception:
        train_fp = None

    manifest = {
        "run_utc": datetime.datetime.utcnow().replace(microsecond=0).isoformat() + "Z",
        "git_head": git_head,
        "dataset": {
            "id": ds_id,
            "revision": ds_rev,
            "train_fingerprint": train_fp,
            "train_rows": int(ds_obj["train"].num_rows) if ds_obj is not None and "train" in ds_obj else None,
        },
        "filters": {
            "start_date": str(globals().get("start_date", "")),
            "end_date_exclusive": str(globals().get("end_date_exclusive", "")),
            "required_years": globals().get("required_years", []),
            "rows_after_window": globals().get("rows_after_window"),
            "tickers_after_window": globals().get("tickers_after_window"),
            "rows_after_full_year": globals().get("rows_after_full_year"),
            "tickers_after_full_year": globals().get("tickers_after_full_year"),
            "rows_after_wrds": globals().get("rows_after_wrds"),
            "tickers_after_wrds": globals().get("tickers_after_wrds"),
        },
        "wrds_meta": {
            "path": wrds_meta_path,
            "sha256": wrds_meta_sha256,
        },
        "output": {
            "rows": int(len(final_dataset_df)),
            "unique_tickers": int(final_dataset_df["ticker"].nunique(dropna=True)),
            "csv": {"path": output_csv, "sha256": csv_sha256},
            "parquet": {"path": output_parquet, "sha256": parquet_sha256},
        },
        "environment": {
            "python": sys.version,
            "platform": platform.platform(),
            "pandas": pd.__version__,
            "datasets": getattr(hf_datasets, "__version__", None),
        },
    }

    manifest_path = "final_dataset_manifest.json"
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)
    print(f"Wrote manifest: {manifest_path}")


Saved final_dataset_df to: final_dataset.csv
Total rows exported: 9432


Saved final_dataset_df to: final_dataset.parquet


Wrote manifest: final_dataset_manifest.json
